In [1]:
from IPython.core.debugger import prompt
!pip install -U langchain-openai

  Using cached uuid_utils-0.12.0-cp39-abi3-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl.metadata (1.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 1.3 MB/s  0:00:00 eta 0:00:01
Using cached uuid_utils-0.12.0-cp39-abi3-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl (603 kB)
  Attempting uninstall: openai
    Found existing installation: openai 1.107.3
    Uninstalling openai-1.107.3:
      Successfully uninstalled openai-1.107.3
  Attempting uninstall: langchain-core━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [openai]
    Found existing installation: langchain-core 1.0.3━━━━━━━━━ 1/4 [openai]
    Uninstalling langchain-core-1.0.3:━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [openai]
      Successfully uninstalled langchain-core-1.0.3━━━━━━━━━━━ 1/4 [openai]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [langchain-openai][langchain-core]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour i

In [3]:
from dotenv import load_dotenv
import os

# load_dotenv()
# API_KEY = os.getenv("OPENAI_API_KEY")
# OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")
# OPEN_MODEL = os.getenv("OPENAI_MODEL")

import os
from dotenv import load_dotenv
load_dotenv()
model_name = os.getenv("LOCAL_MODE")
base_url = os.getenv("LOCAL_BASE_URL")

In [4]:
# from langchain_openai import ChatOpenAI
#
# llm = ChatOpenAI(
#     model=OPEN_MODEL,
#     api_key=API_KEY,
#     base_url=OPENAI_BASE_URL,
#     temperature=0.2,      # 稳定输出
#     timeout=1200,         # 超时保护（秒）
#     max_retries=2         # 简单重试
# )
from langchain_ollama import OllamaLLM
llm = OllamaLLM(
    model = model_name,
    base_url=base_url,
    temperature=0.2
)

In [5]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
prompt = PromptTemplate(
    input_variables=["topic"],
    template="请以 {topic} 为主题，写一首简短的诗"
)

In [7]:
chain = LLMChain(
    llm = llm,
    prompt = prompt,
    verbose = True # 显示调用醒醒
)

result = chain.invoke({"topic":"冬天"})

print(result)



> Entering new LLMChain chain...
Prompt after formatting:
请以 冬天 为主题，写一首简短的诗

> Finished chain.
{'topic': '冬天', 'text': "Here is a short poem with the theme of winter:\n\nWinter's chill begins to bite,\nFrosty mornings, dark and bright.\nSnowflakes swirl, and dance, and play,\nAs earth and sky are wrapped in gray.\n\nThe fireplace crackles warm and low,\nA haven from the cold outside to go.\nHot chocolate warms the hands and heart,\nAs winter's peaceful silence takes its part."}


In [9]:
from langchain_classic.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# 1.定定义一个数据机构
class BookReview(BaseModel):
    title: str = Field(description="书名")
    author: str = Field(description="作者")
    rating: int = Field(description="评分 1-5")
    summary: str = Field(description="摘要")


# 2. 创建解析器
parser = PydanticOutputParser(pydantic_object=BookReview)

# 3. 创建提示，要求 AI 严格输出格式
prompt = PromptTemplate(
    template="请评价以下书籍：{book_info}\n\n{format_instructions}",
    input_variables=["book_info"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)
chain = LLMChain(llm=llm, prompt=prompt, output_parser=parser)

# 4. 运行链
result = chain.invoke({"book_info":"《三体》，刘慈欣的科幻小说"})
print(result)

{'book_info': '《三体》，刘慈欣的科幻小说', 'text': BookReview(title='_三体_', author='刘慈欣 (Liu Cixin)', rating=4, summary="《三体》 is a science fiction novel that explores the first contact between humans and an alien civilization. The story is set against the backdrop of China's Cultural Revolution and the Three-Body Problem, which refers to the chaotic and unpredictable nature of the aliens' planet. The novel delves into themes of science, technology, and humanity's place in the universe.")}
